# Datathon26 — datasets & evaluators demo

At the datathon, models don't run locally. You submit studies to an external
**reportbench** service, which returns pre-computed outputs. The
`radharmony.datathon26` submodule scores those outputs:

- **Classification** — the service returns one `embedding.npy` per study; a
  probe evaluator scores it (in place of running an image encoder).
- **Report generation** — the service returns a generated `report.txt`; RadEval
  scores it against the reference report.

The evaluators are thin subclasses of the standard RadHarmony ones, so nothing
is lost — patient-grouped k-fold, bootstrap, `macro_average`, the full RadEval
suite, `generations.csv`, and so on all still work.

> **This demo is self-contained** — it fabricates a tiny batch table and fake
> result files so it runs end-to-end without the service. Sections 5–6 show how
> to point the same code at real data and how to submit a run yourself.

## The workflow this fits into

```
sample_data.ipynb            reportbench service            radharmony.datathon26
-----------------            -------------------            ---------------------
sample + anonymize    -->  submit run1/ (per-study    -->  point a dataset at the
5 CXR datasets             image folders)                  results dir + batch df
                                                                   |
dataset_{A..E}.csv         writes per study:                       v
(labels + reports)           <study>/embedding.npy          evaluate() -> metrics
mapping.csv (private)        <study>/report.txt
```

You provide the **batch table** (labels + reference reports) and the **results
dir** (what the service returned). A dataset joins them; an evaluator scores.

## 0. Setup

In [1]:
import os
import tempfile

import numpy as np
import pandas as pd

import radharmony.datathon26  # noqa: F401  -> registers all datathon26_* keys
from radharmony.datathon26 import (
    EmbeddingResultsDataset,
    ReportResultsDataset,
    DatathonLinearProbeEvaluator,
    DatathonReportGenerationEvaluator,
)
from radharmony.registry import list_datasets
from radharmony.evaluator._registry import list_evaluators

print("datasets: ", sorted(k for k in list_datasets() if "datathon" in k))
print("evaluators:", sorted(k for k in list_evaluators() if "datathon" in k))

datasets:  ['datathon26_embedding', 'datathon26_report']
evaluators: ['datathon26_knn_probe', 'datathon26_linear_probe', 'datathon26_prototype_probe', 'datathon26_report_generation', 'datathon26_svm_probe']


## 1. The batch table

In production this is one of the anonymized `dataset_{A..E}.csv` frames — one
row per study, carrying the anonymized `patient_id` / `study_id`, the one-hot
finding **labels**, and the inline reference **report**. Below is a tiny
synthetic stand-in (a few patients, 3 labels).

The string columns (`patient_sex`, `study_date`) are deliberate: the embedding
dataset must auto-detect labels as binary 0/1 columns only and leave these out.

In [2]:
rng = np.random.default_rng(0)
EMB_DIM = 32

rows = []
for p in range(10):
    pid = f"A_patient_{p:05d}"
    for s in range(rng.integers(2, 5)):  # a few studies per patient
        sid = f"{pid}_study_{s:05d}"
        cardio = int(rng.integers(0, 2))
        rows.append(
            {
                "patient_id": pid,
                "study_id": sid,
                "image_path": f"{sid}/frontal.dcm",  # original path (unused here)
                "view_position": "PA",
                "report": "FINDINGS: cardiomegaly with small pleural effusion.\n"
                "IMPRESSION: abnormal cardiac silhouette.",
                "patient_sex": "F",  # string extra (not a label)
                "study_date": "2026-01-01",  # string extra (not a label)
                "cardiomegaly": cardio,
                "effusion": int(rng.integers(0, 2)),
                "pneumonia": int(rng.integers(0, 2)),
            }
        )

batch_df = pd.DataFrame(rows)
print(f"{len(batch_df)} studies across {batch_df.patient_id.nunique()} patients")
batch_df.head()

29 studies across 10 patients


,patient_id,study_id,image_path,view_position,report,patient_sex,study_date,cardiomegaly,effusion,pneumonia
0,A_patient_00000,A_patient_00000_study_00000,A_patient_00000_study_00000/frontal.dcm,PA,FINDINGS: cardiomegaly with small pleural effu...,F,2026-01-01,1,1,0
1,A_patient_00000,A_patient_00000_study_00001,A_patient_00000_study_00001/frontal.dcm,PA,FINDINGS: cardiomegaly with small pleural effu...,F,2026-01-01,0,0,0
2,A_patient_00000,A_patient_00000_study_00002,A_patient_00000_study_00002/frontal.dcm,PA,FINDINGS: cardiomegaly with small pleural effu...,F,2026-01-01,0,0,1
3,A_patient_00000,A_patient_00000_study_00003,A_patient_00000_study_00003/frontal.dcm,PA,FINDINGS: cardiomegaly with small pleural effu...,F,2026-01-01,1,1,1
4,A_patient_00001,A_patient_00001_study_00000,A_patient_00001_study_00000/frontal.dcm,PA,FINDINGS: cardiomegaly with small pleural effu...,F,2026-01-01,1,1,1


## 2. Fake the reportbench results

We write a random `embedding.npy` (nudged by the label so the probe isn't
degenerate) and a generated `report.txt` per study into a temp directory — the
shape the real service returns. **With real data you skip this cell** and point
straight at the folder the service gave you.

In [3]:
RESULTS_DIR = tempfile.mkdtemp(prefix="reportbench_results_")

for _, r in batch_df.iterrows():
    study_dir = os.path.join(RESULTS_DIR, r["study_id"])
    os.makedirs(study_dir, exist_ok=True)
    # embeddings task: shift the vector by the label so it carries signal
    vec = rng.normal(r["cardiomegaly"] * 1.5, 1.0, size=EMB_DIM).astype("float32")
    np.save(os.path.join(study_dir, "embedding.npy"), vec)
    # report task: the model-generated report
    with open(os.path.join(study_dir, "report.txt"), "w") as f:
        f.write(
            "FINDINGS: mild cardiomegaly, no effusion.\n"
            "IMPRESSION: mildly enlarged heart."
        )

print("results dir:", RESULTS_DIR)
example = batch_df["study_id"].iloc[0]
print("layout for one study:")
for fn in sorted(os.listdir(os.path.join(RESULTS_DIR, example))):
    print(f"  {example}/{fn}")

results dir: /tmp/reportbench_results_3307dqrc
layout for one study:
  A_patient_00000_study_00000/embedding.npy
  A_patient_00000_study_00000/report.txt


## 3. Classification — embeddings → probe

`EmbeddingResultsDataset` loads each `<study>/embedding.npy` into `img` and the
one-hot labels into `cls`. Labels are auto-detected as the binary 0/1 columns,
so `patient_sex` / `study_date` stay out.

`DatathonLinearProbeEvaluator` uses a pass-through encoder, so the pre-extracted
embeddings feed straight into per-label logistic regression. (`device='cpu'`,
`autocast_dtype=None` keep the demo GPU-free.)

In [4]:
cls_ds = EmbeddingResultsDataset(RESULTS_DIR, harmonized_df=batch_df)
print("auto-detected labels:", cls_ds.LABEL_COLS)

cls_eval = DatathonLinearProbeEvaluator(
    dataset=cls_ds,
    n_folds=3,
    device="cpu",
    autocast_dtype=None,
    num_workers=1,
    batch_size=16,
)
cls_results = cls_eval.evaluate()
cls_results.head(12)

auto-detected labels: ['cardiomegaly', 'effusion', 'pneumonia']


extracting embeddings:   0%|          | 0/2 [00:00<?, ?batch/s]

,n_train,fold,seed,bootstrap,label,auroc,auprc,f1,accuracy,balanced_accuracy,tpr,tnr,ppv,npv,mcc,threshold
0,-1,0,-1,-1,cardiomegaly,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.719404
1,-1,0,-1,-1,effusion,0.433333,0.625132,0.750000,0.636364,0.600000,1.000000,0.200000,0.600000,1.000000,0.346410,0.356106
2,-1,0,-1,-1,pneumonia,0.366667,0.653078,0.500000,0.636364,0.666667,0.333333,1.000000,1.000000,0.555556,0.430331,0.658585
3,-1,1,-1,-1,cardiomegaly,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.650738
4,-1,1,-1,-1,effusion,0.857143,0.961735,0.833333,0.777778,0.857143,0.714286,1.000000,1.000000,0.500000,0.597614,0.525003
5,-1,1,-1,-1,pneumonia,0.555556,0.766270,0.727273,0.666667,0.666667,0.666667,0.666667,0.800000,0.500000,0.316228,0.479558
6,-1,2,-1,-1,cardiomegaly,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.709335
7,-1,2,-1,-1,effusion,0.888889,0.948413,0.923077,0.888889,0.833333,1.000000,0.666667,0.857143,1.000000,0.755929,0.520016
8,-1,2,-1,-1,pneumonia,0.400000,0.433333,0.666667,0.555556,0.600000,1.000000,0.200000,0.500000,1.000000,0.316228,0.461131
9,-1,0,-1,-1,macro_average,0.600000,0.759404,0.750000,0.757576,0.755556,0.777778,0.733333,0.866667,0.851852,0.592247,0.578032


Per-label AUROC/AUPRC/F1 plus a `macro_average` row per fold. Averaging across
folds gives the headline number:

In [5]:
macro = cls_results[cls_results["label"] == "macro_average"]
print("mean macro AUROC across folds:", round(macro["auroc"].mean(), 4))
cls_results.groupby("label")[["auroc", "auprc", "f1"]].mean().round(4)

mean macro AUROC across folds: 0.7224


,auroc,auprc,f1
label,,,
cardiomegaly,1.0000,1.0000,1.0000
effusion,0.7265,0.8451,0.8355
macro_average,0.7224,0.8209,0.8223
pneumonia,0.4407,0.6176,0.6313


> Swap in `DatathonKNNProbeEvaluator`, `DatathonSVMProbeEvaluator`, or
> `DatathonPrototypeProbeEvaluator` — identical call shape.

## 4. Report generation — reference + predicted → RadEval

`ReportResultsDataset` loads the generated `<study>/report.txt` into `img` and
keeps the inline reference in `report`. `DatathonReportGenerationEvaluator`
uses a pass-through generator and scores predicted-vs-reference with RadEval,
both parsed to the same `ref_section` (findings here).

We use `LIGHT_METRICS` (bleu / rouge / bertscore / radgraph / f1chexbert) for
speed; the default is the full 16-metric suite.

> Needs the `radeval` dependency; the first run downloads the metric models.

In [6]:
from radharmony.evaluator.metrics.language import LIGHT_METRICS

rep_ds = ReportResultsDataset(RESULTS_DIR, harmonized_df=batch_df)

rep_eval = DatathonReportGenerationEvaluator(
    dataset=rep_ds,
    metrics=LIGHT_METRICS,
    ref_section="findings",
    num_workers=1,
    batch_size=16,
)
rep_results = rep_eval.evaluate()
rep_results

scoring reports:   0%|          | 0/2 [00:00<?, ?batch/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using device: cuda:0


Deprecated in 0.9.0: WordPiece.__init__ will not create from files anymore, try `WordPiece.from_file` instead


Output()

,fold,seed,bootstrap,label,bleu,rouge1,rouge2,rougeL,bertscore,radgraph_simple,radgraph_partial,radgraph_complete,f1chexbert_5_micro_f1,f1chexbert_all_micro_f1,f1chexbert_5_macro_f1,f1chexbert_all_macro_f1,f1chexbert_5_weighted_f1,f1chexbert_all_weighted_f1
0,-1,-1,-1,report,0.0,0.4444,0.0,0.4444,0.6667,0.2857,0.2857,0.2857,0.6667,0.6667,0.2,0.0714,0.5,0.5


The per-study predicted/reference pairs that were scored:

In [7]:
rep_eval._records_dataframe().head()

,sample_id,reference,hypothesis
0,A_patient_00000_study_00000,cardiomegaly with small pleural effusion.,"mild cardiomegaly, no effusion."
1,A_patient_00000_study_00001,cardiomegaly with small pleural effusion.,"mild cardiomegaly, no effusion."
2,A_patient_00000_study_00002,cardiomegaly with small pleural effusion.,"mild cardiomegaly, no effusion."
3,A_patient_00000_study_00003,cardiomegaly with small pleural effusion.,"mild cardiomegaly, no effusion."
4,A_patient_00001_study_00000,cardiomegaly with small pleural effusion.,"mild cardiomegaly, no effusion."


In [ ]:
import shutil

shutil.rmtree(RESULTS_DIR, ignore_errors=True)  # clean up the fabricated demo fixtures
print("cleaned up", RESULTS_DIR)

cleaned up /tmp/reportbench_results_3307dqrc


## 5. Pointing at real data

The fake files in §2 used the dataset defaults (`<study>/embedding.npy`,
`<study>/report.txt`). The **real service** uses a different per-study layout,
under `_reportbench_out/<model>/`:

```
A_patient_00018_study_00001/
    model-a_pa.npz          # one embedding PER IMAGE: <model>_<stem>.npz
    model-a_lateral.npz     #   (an archive holding `global` [dim] + `patches`)
    model-a_report.txt      # one report per study: <model>_report.txt
```

Two things change for real data:

1. **Pass a `result_path=` resolver** so the dataset finds the real filenames —
   `viewwise_embedding_path(model)` (defaults to `.npz`) and
   `perstudy_report_path(model)` build them. `EmbeddingResultsDataset` reads the
   `global` vector out of each `.npz` automatically.
2. **Match the granularity.** Embeddings are per-image → keep one row per image.
   Reports are per-study → `drop_duplicates("study_id")` first.

The cells below load your finished `model-a` run for dataset A. `manifest.json`
lists which studies/images were submitted; we filter the full `dataset_A.csv`
down to those. The reader accepts the **study-keyed** schema
(`{"studies": {"<id>": {"images": [...], "indication"?}}}`, `path` = bare
filename) and the older flat `{"images": [{"path": "<id>/<file>", ...}]}` form.

Registry keys (for `resolve_dataset` / `resolve_evaluator`):
`datathon26_embedding`, `datathon26_report`, `datathon26_linear_probe`,
`datathon26_knn_probe`, `datathon26_svm_probe`, `datathon26_prototype_probe`,
`datathon26_report_generation`.

In [ ]:
import json

# Your team's finished run for dataset A (paths from INSTRUCTIONS.md).
RUN_DIR = (
    "/mnt/NAS4/projects/bkhosra/sharing/datathon/data_deposition/"
    "a8110d864837b9693cbdb69b/dt26_runs/dataset_A"
)
CSV_PATH = "/mnt/NAS4/projects/fli40_sharing/datathon26_cxr_datasets/dataset_A.csv"
MODEL = "model-a"
OUT_DIR = f"{RUN_DIR}/_reportbench_out/{MODEL}"  # <study>/<model>_<stem>.npz + <model>_report.txt

# manifest.json lists exactly which images were submitted for this run. Accept
# both the study-keyed schema ({"studies": {"<id>": {"images": [{"path","view"}],
# ...}}}, path = bare filename) and the older flat one ({"images": [{"path":
# "<id>/<file>", ...}]}); normalise to path="<study>/<file>".
with open(f"{RUN_DIR}/manifest.json") as f:
    manifest = json.load(f)

if "studies" in manifest:
    rows = [
        {"path": f"{sid}/{img['path']}", "view": img.get("view")}
        for sid, spec in manifest["studies"].items()
        for img in spec["images"]
    ]
else:
    rows = manifest["images"]
sub = pd.DataFrame(rows)  # columns: path ("<study>/<view>.dcm"), view
print(f"{len(sub)} submitted images")
sub.head()

In [ ]:
# Filter dataset_A to the studies actually submitted (join on the manifest paths).
# Lowercase BOTH sides: manifest paths are mixed-case ("A_patient_..."), so
# lowercasing only image_path would never match and the filter returns nothing.
df_A_sub = pd.read_csv(CSV_PATH)
df_A_sub = df_A_sub[df_A_sub.image_path.str.lower().isin(sub.path.str.lower())].copy()

print(f"{len(df_A_sub)} image-rows across {df_A_sub.study_id.nunique()} studies")
df_A_sub.head()

821 image-rows across 500 studies


,patient_id,study_id,image_path,view_position,report,atelectasis,cardiomegaly,consolidation,edema,enlarged_cardiomediastinum,fracture,lung_lesion,lung_opacity,no_finding,pleural_effusion,pleural_other,pneumonia,pneumothorax,support_devices
30,A_patient_00018,A_patient_00018_study_00001,A_patient_00018_study_00001/lateral.dcm,LATERAL,FINAL REPORT\...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
31,A_patient_00018,A_patient_00018_study_00001,A_patient_00018_study_00001/pa.dcm,PA,FINAL REPORT\...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
105,A_patient_00058,A_patient_00058_study_00001,A_patient_00058_study_00001/ap.dcm,AP,FINAL REPORT\...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
193,A_patient_00111,A_patient_00111_study_00001,A_patient_00111_study_00001/lateral.dcm,LATERAL,FINAL REPORT\...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
194,A_patient_00111,A_patient_00111_study_00001,A_patient_00111_study_00001/pa.dcm,PA,FINAL REPORT\...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
from radharmony.datathon26 import (
    EmbeddingResultsDataset,
    ReportResultsDataset,
    viewwise_embedding_path,
    perstudy_report_path,
)

# Classification — one embedding per view -> <study>/<model>_<view>.npy.
# df_A_sub already has one row per image, which is the right granularity here.
cls_ds = EmbeddingResultsDataset(
    OUT_DIR,
    harmonized_df=df_A_sub,
    result_path=viewwise_embedding_path(MODEL),
)
cls_results = DatathonLinearProbeEvaluator(
    dataset=cls_ds, n_folds=5, device="cpu", autocast_dtype=None, num_workers=2
).evaluate()
cls_results.groupby("label")[["auroc", "auprc", "f1"]].mean().round(4)

This process (pid=449689) is multi-threaded, use of fork() may lead to deadlocks in the child.
This process (pid=449689) is multi-threaded, use of fork() may lead to deadlocks in the child.


extracting embeddings:   0%|          | 0/13 [00:00<?, ?batch/s]

,auroc,auprc,f1
label,,,
atelectasis,0.8200,0.4029,0.4897
cardiomegaly,0.8152,0.3895,0.4270
consolidation,0.7635,0.1234,0.1352
edema,0.9271,0.4410,0.3356
enlarged_cardiomediastinum,0.8487,0.1640,0.2541
fracture,0.5447,0.0857,0.1953
lung_lesion,0.5886,0.1697,0.1217
lung_opacity,0.7619,0.2864,0.2981
macro_average,0.8019,0.3973,0.4054


In [ ]:
from radharmony.evaluator.metrics.language import LIGHT_METRICS

# Report generation — one report per study -> <study>/<model>_report.txt.
# Reports are per-study, so dedupe df_A_sub to one row per study.
rep_ds = ReportResultsDataset(
    OUT_DIR,
    harmonized_df=df_A_sub.drop_duplicates("study_id").copy(),
    result_path=perstudy_report_path(MODEL),
)
DatathonReportGenerationEvaluator(
    dataset=rep_ds, metrics=LIGHT_METRICS, ref_section="findings", num_workers=2
).evaluate()

This process (pid=449689) is multi-threaded, use of fork() may lead to deadlocks in the child.


scoring reports:   0%|          | 0/63 [00:00<?, ?batch/s]

## 6. One-shot: raw images → dataset via the service

Sections 3–5 start from a results folder. The **one-shot factories** do the
submission for you: give them a harmonized frame and the raw image root — *the
same two inputs any RadHarmony dataset takes* — and `DatathonEmbeddingDataset` /
`DatathonReportDataset` stage the studies, drive `rbclient.py`
(`config → prepare → submit → watch`) through `ReportBenchClient`, and return a
ready result dataset.

Fill in your real values (from your team `INSTRUCTIONS.md`) and run this **on the
datathon server** — it submits real jobs, so it needs the server and your API
key. It's a template, not "Run All"-safe.

> **Wired to the live format.** The factories submit with the real task names
> (`embed` / `report`), point the dataset at
> `_reportbench_out/<model>/` with the matching `result_path`, and verify the
> real output filenames (`<model>_<stem>.npz`, `<model>_report.txt`) before
> returning — `EmbeddingResultsDataset` reads `global` out of each `.npz`.
> - **Indication:** pass `indication_col=` to `DatathonReportDataset` to attach
>   each study's indication; it is written into a study-keyed `manifest.json`
>   (the only channel for indications — there is no `text.txt`). The `embed` task
>   submits without a manifest.
> - **Staging cost:** `prepare` writes real file copies into the run folder (so
>   the manifest's image paths resolve inside it), i.e. one copy of the images.

In [ ]:
from radharmony.datathon26 import (
    ReportBenchClient,
    DatathonEmbeddingDataset,
    DatathonReportDataset,
)

# --- fill these in (see your team INSTRUCTIONS.md), then run on the server ---
RBCLIENT_PATH = "/mnt/NAS4/projects/bkhosra/sharing/datathon/skill/rbclient.py"
DATA_DIR = "/mnt/NAS4/projects/bkhosra/sharing/datathon/data_deposition/<team_id>"
API_KEY = "rb_..."
BASE_IMAGE_DIR = "/path/to/raw/images"  # what image_path in batch_df is relative to
MODEL = "model-a"

client = ReportBenchClient(
    rbclient_path=RBCLIENT_PATH, data_dir=DATA_DIR, api_key=API_KEY
)

# Classification — stage batch_df's studies, submit the embed task, wait, score.
# batch_df should have one row per image (per view); the .npz `global` vector feeds the probe.
emb_ds = DatathonEmbeddingDataset(
    batch_df, BASE_IMAGE_DIR, client=client, model=MODEL, run_name="run1"
)
DatathonLinearProbeEvaluator(dataset=emb_ds, n_folds=5).evaluate()

# Report generation — same flow, report task. Pass indication_col= to attach each
# study's indication (written into the job manifest); omit it to report from images alone.
rep_ds = DatathonReportDataset(
    batch_df, BASE_IMAGE_DIR, client=client, model=MODEL, run_name="run1_report",
    # indication_col="indication",
)
DatathonReportGenerationEvaluator(
    dataset=rep_ds, metrics=LIGHT_METRICS, ref_section="findings"
).evaluate()